In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Cellule 1 : Installation et Chargement (CORRIGÉE)

# 1. D'ABORD : On installe les librairies (C'est l'étape cruciale)
print("⏳ Installation de Unsloth et des dépendances...")
# On force l'installation immédiatement
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes

# 2. ENSUITE : On peut importer sans erreur
import torch
from unsloth import FastLanguageModel

# 3. ENFIN : On charge le modèle
print("⏳ Chargement du modèle Qwen en mémoire GPU...")
max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-14B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# On se met en mode "Inférence" pour la génération de données
FastLanguageModel.for_inference(model)
print("✅ C'EST BON ! Modèle chargé et prêt.")

⏳ Installation de Unsloth et des dépendances...
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-5sn7ozcx/unsloth_82b8a2b614c84805a40df04e1926cfa5
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-5sn7ozcx/unsloth_82b8a2b614c84805a40df04e1926cfa5
  Resolved https://github.com/unslothai/unsloth.git to commit f3f9d148874389996636e4fd110c7255b9f62c77
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

✅ C'EST BON ! Modèle chargé et prêt.


In [ ]:
# Cellule 2 : Génération du Dataset d'entraînement
import json
import glob
import os
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/drive')

# --- CONFIGURATION (Mets le bon chemin ici !) ---
dossier_txt = "/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Données/données en txt"
output_file = "dataset_entrainement.jsonl"
# ------------------------------------------------

files = glob.glob(f"{dossier_txt}/*.txt")
print(f"📂 Fichiers TXT trouvés : {len(files)}")

if len(files) == 0:
    print("❌ ERREUR : Pas de fichiers trouvés. Vérifie le chemin !")
else:
    print("🚀 Génération des questions en cours...")

    prompt_generation = """Tu es un expert RH du CNRS.
    Analyse le texte ci-dessous et crée 3 paires Question/Réponse pour l'entraînement.

    1. Une question technique précise.
    2. Une question sur le contexte (labo, ville).
    3. Une question administrative.

    Réponds UNIQUEMENT avec ce JSON strict :
    [
      {"instruction": "Question...", "input": "Contexte...", "output": "Réponse..."},
      {"instruction": "...", "input": "...", "output": "..."}
    ]
    """

    with open(output_file, 'w', encoding='utf-8') as f_out:
        for file_path in tqdm(files):
            try:
                # Lecture
                with open(file_path, 'r', encoding='utf-8') as f:
                    content = f.read()[:2500] # On garde le début

                # Génération
                messages = [
                    {"role": "system", "content": prompt_generation},
                    {"role": "user", "content": f"Texte :\n{content}"}
                ]
                inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

                outputs = model.generate(inputs, max_new_tokens=500, temperature=0.7)
                response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

                # Extraction JSON
                json_part = response.split("assistant\n")[-1]
                start, end = json_part.find("["), json_part.rfind("]") + 1
                if start != -1 and end != -1:
                    data = json.loads(json_part[start:end])
                    for item in data:
                        item['input'] = content # On garde le contexte
                        json.dump(item, f_out, ensure_ascii=False)
                        f_out.write('\n')
            except: pass

    print(f"✅ Dataset généré : {output_file}")